# Module 1 — Understand the benchmark before building retrieval

**Learning objective:** inspect FinanceBench's questions, documents, and evidence; understand what an evaluation can legitimately measure.

You will load a pinned local corpus, inspect company coverage, audit AMD's references, and explore another company. The same Python functions will support later evaluation and application code.

This notebook inspects benchmark annotations. PDF text extraction begins in Module 2. No API keys or model calls are needed.

## 1. Prepare once, then run offline

From the repository root:

```powershell
uv sync --locked
uv run python -m finsight.financebench fetch --company AMD
uv run python -m ipykernel install --sys-prefix --name finsight --display-name "Python (finsight)"
uv run jupyter lab
```

Select **Python (finsight)** and run all cells. Fetching is an explicit preparation step. The cells below read cached files without network access. `FINSIGHT_DATA_DIR` can point to another cache, including authored synthetic fixtures used by our notebook test.

In [ ]:
import json
import os
from collections import Counter
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

from finsight.financebench import audit_dataset, load_dataset

cwd = Path.cwd()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file())
DATA_DIR = Path(os.environ.get("FINSIGHT_DATA_DIR", PROJECT_ROOT / "data/financebench"))
COMPANY = "AMD"  # Change this to explore another company's annotations.
dataset = load_dataset(DATA_DIR, company=COMPANY)
print(f"Selected company: {dataset.company}")
print(f"Local cache: {DATA_DIR}")

## 2. Inspect coverage

The public release is a sample, not the full FinanceBench benchmark. The number of questions, filings, and annotated pages are different quantities. A company with several questions can still have very limited document or year coverage.

Inspect the distribution before deciding how much confidence to place in a score.

In [ ]:
coverage = Counter(q.company for q in dataset.all_questions)
print(f"Public questions in this cache: {len(dataset.all_questions)}")
print(f"Companies represented: {len(coverage)}")
print(f"Metadata rows: {len(dataset.all_documents)}")
pprint([{"company": name, "questions": count} for name, count in coverage.most_common(12)])

## 3. Follow question → document → evidence

A question names an intended filing. Each evidence annotation also names a document, and can refer to a different filing. Resolve both rather than assuming they always match.

Gold answers and evidence are evaluation annotations. Future retrieval and generation will consume the uploaded PDF content and the user's question. These gold annotations must stay outside that input path.

In [ ]:
pprint(
    [
        {
            "id": q.financebench_id,
            "document": q.doc_name,
            "question_type": q.question_type,
            "reasoning": q.question_reasoning,
        }
        for q in dataset.questions
    ]
)
print("Resolved filing metadata:")
pprint(
    [
        {"document": d.doc_name, "period": d.doc_period, "type": d.doc_type}
        for d in dataset.documents
    ]
)

In [ ]:
first = dataset.questions[0]
print("Question:", first.question)
print("Gold answer:", first.answer)
print("Optional justification:", first.justification)
print("Evidence annotations:")
pprint(
    [
        {
            "document": e.doc_name,
            "page_index": e.evidence_page_num,
            "display_page": e.evidence_page_num + 1,
            "text_preview": e.evidence_text[:250],
        }
        for e in first.evidence
    ]
)

**Source details matter.** Actual evidence records use `doc_name`, although the upstream README describes `evidence_doc_name`. Question IDs are strings. Reasoning and justification can be null; preserve those missing labels.

Evidence page indexes are **zero-based**. `display_page = page_index + 1` means the physical PDF page position, which may differ from the page number printed on the document. We have not checked page bounds or extracted PDF text yet.

In [ ]:
evidence_rows = [
    {
        "question_id": q.financebench_id,
        "document": e.doc_name,
        "page_index": e.evidence_page_num,
        "display_page": e.evidence_page_num + 1,
    }
    for q in dataset.questions
    for e in q.evidence
]
pprint(evidence_rows)
print("Missing reasoning labels:", sum(q.question_reasoning is None for q in dataset.questions))

## 4. Audit readiness and provenance

The audit checks references, duplicates, metadata ambiguity, file presence, and recorded hashes. A warning can flag a problem in an unrelated catalogue entry without invalidating the selected AMD corpus. Required ambiguous references are errors.

`ok` means no audit errors; also inspect `provenance_verified`. A manually prepared cache without a manifest can pass annotation checks while its provenance remains unverified. A hash verifies consistency with the recorded download, not the truth of an answer or the completeness of a PDF.

In [ ]:
report = audit_dataset(dataset)
pprint(asdict(report))
print("Annotation/corpus audit OK:", report.ok)
print("Pinned cache verified:", report.provenance_verified)

For the pinned real AMD sample, expect **8 questions, 2 filings, 9 evidence annotations, 8 distinct document-page locations, and 2 missing reasoning labels**. The catalogue also contains conflicting metadata for `FOOTLOCKER_2023_annualreport`, outside this AMD selection.

Synthetic fixtures intentionally have different counts. The code computes counts from the supplied data rather than hard-coding the expected AMD result.

## 5. Small experiment — change the company

Choose another company below. The two JSONL files contain annotations across companies, so you can inspect their coverage offline. Its PDFs have not necessarily been fetched. Audit findings should make that distinction visible.

To prepare a complete cache for that company later, run `fetch --company "COMPANY NAME"` explicitly outside this notebook. Reusing the cache retains existing company selections.

In [ ]:
OTHER_COMPANY = next((name for name, _ in coverage.most_common() if name != dataset.company), None)
if OTHER_COMPANY is not None:
    other = load_dataset(DATA_DIR, company=OTHER_COMPANY)
    other_report = audit_dataset(other)
    pprint(
        {
            "company": other.company,
            "questions": other_report.question_count,
            "referenced_documents": other_report.document_count,
            "missing_pdfs": sum(f.code == "missing_pdf" for f in other_report.findings),
            "provenance_verified": other_report.provenance_verified,
        }
    )
else:
    print("This cache contains annotations for only one company.")

Try setting `OTHER_COMPANY` to a different name shown in the coverage table. Compare:

- Questions per document and the years represented.
- Missing reasoning labels and the kinds of reasoning described.
- Known evidence locations versus the PDFs actually present.

**Reflect:** would an improvement on eight AMD examples justify a claim about arbitrary company filings? What additional unseen documents and independently reviewed questions would you need?

## 6. What these labels can support

| Question | Evidence available now | Evaluation implication |
|---|---|---|
| Did retrieval find known supporting locations? | Annotated document/page/text references | Map them to parsed content first. Known-evidence coverage is measurable; annotations are not exhaustive relevance judgments. |
| What about Recall@k and MRR? | No exhaustive chunk-level relevance set | Define the candidate unit and valid relevance mapping before reporting these metrics. Unannotated passages may also be relevant. |
| Is an answer correct? | Gold answers and sometimes justification | Use a documented correctness rubric, including units and periods. Human judgments and LLM-judge estimates must be labeled separately. |
| Is an answer grounded and cited correctly? | Supporting text and page references | Check claims against actual retrieved source content. Correct-looking citations alone do not establish support. |
| Does an approach generalize? | A small, inspected development sample | Reserve independently reviewed unseen cases; do not call this inspected sample a held-out test set. |

Module 1 reports data and readiness counts. Retrieval metrics, answer metrics, latency, and cost measurements begin when their pipeline stages exist.

## Module review checkpoint

You should now be able to explain the source schema, reproduce the selected corpus, interpret audit findings, and distinguish annotated support from exhaustive retrieval relevance.

Next module, after review: parse the PDFs while preserving source page locations and inspect extraction quality, especially financial tables.

In [ ]:
module1_summary = {
    "company": report.company,
    "questions": report.question_count,
    "documents": report.document_count,
    "evidence_annotations": report.evidence_count,
    "unique_evidence_pages": report.unique_evidence_page_count,
    "missing_reasoning_labels": report.missing_reasoning_count,
    "annotation_audit_ok": report.ok,
    "provenance_verified": report.provenance_verified,
}
print(json.dumps(module1_summary, sort_keys=True))